### 1. 환경 설정

In [ ]:
!pip install -r requirements.txt

- tcpdump 
    - Linux 환경에서 가장 표준적으로 사용하는 CLI Packet Sniffer이다.
    - 특정 인터페이스를 지나다니는 실시간 패킷을 감시하고 캡쳐한다.
    - IP, 포트번호, 프로토콜 조건을 걸어 필요한 패킷만 필터링해서 보여준다.
    - 조회하는 것에만 기능이 있고, 새로운 패킷을 임의로 만들어서 전송하는 기능은 없다.
- Scapy
    - Python 환경에서 네트워크 패킷을 직접 제작, 수정, 송수신, 디코딩할 수 있게 해준다.
    - 소프트웨어 라우팅 로직을 파이썬 코드로 직접 작성할 수 있다. 

### 2. 가상 네트워크 인터페이스(veth) 생성 및 활성화

In [ ]:
sudo ip link add veth-a type veth peer name veth-b
sudo ip link set veth-a up
sudo ip link set veth-b up
sudo ip addr add 10.0.0.1/24 dev veth-a
sudo ip addr add 10.0.0.2/24 dev veth-b

- 가상 네트워크 인터페이스 2개를 생성한다. (veth-a, veth-b)
- ip addr add (부여할 ip 주소) (이 ip 주소를 부여할 대상 네트워크 장치/인터페이스 이름)
    - ip: iproute2 라이브러리에 포함된 리눅스 네트워크 설정을 제어하는 명령어
    - addr: ip 주소 관련 작업을 수행하겠다는 의미
    - add: 새로운 ip 주소를 추가하겠다는 동작 지정
- veth만 생성하면 가상 랜카드를 심은 후이고, ip 할당을 통해 통신 가능한 상태로 만든다.

In [ ]:
ip addr show veth-a

### 3. ip 할당의 의미

- 랜카드에 위치(IP)를 할당한다는 것은 리눅스 커널의 패킷 처리 방식을 변경한다.
- 동일 서브넷으로 인식할 때 (10.0.0.1/24)
- 커널은 같은 L2 영역에 있다고 인식한다.
- 라우터(게이트웨이)를 거치지 않고 ARP를 실행하여 상대방의 MAC 주소를 직접 찾은 뒤 패킷을 바로 전송할 수 있다.

![Image](./fig/4_1.png)

- IP 주소와 서브넷 마스크를 지정하면 리눅스 커널은 내부에 다음과 같은 라우팅 규칙을 자동으로 등록한다.
- Destination, Gateway, Genmask, Flags, Iface
    - 위 5가지의 규칙을 기록하여 10.0.0.0 ~ 10.0.0.255 사이의 모든 패킷은 별도의 게이트웨이 없이 특정 인터페이스로 보내라는 의미이다.
    - 게이트웨이를 거치지 않는다는 것은 0.0.0.0으로 설정하면 된다.

- 서브넷 마스크를 24로 설정하면 해당 대역의 마지막 주소인 10.0.0.255가 broadcast 주소로 자동 지정된다.
- 10.0.0.1 ~ 10.0.0.255 사이의 ip로 패킷을 보내면 동일한 클래스의 네트워크에 묶여있는 모든 인터페이스가 그 패킷을 수신한다.
- 같은 서브넷 마스크를 가지고 있어야, 가상 랜선을 통해서 1:1로 패킷을 직접 주고 받을 수 있다.
    - 그렇지 않으면 라우팅이 필요해서 게이트웨이를 거쳐야한다.

### 4. 네트워크 인터페이스 생성 확인

In [ ]:
ip addr show veth-a

- 특정 네트워크 인터페이스의 상세 정보 및 현재 상태를 조회한다.
- 결과값
    - 4: veth-a@veth-b: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500 qdisc noqueue state UP group default qlen 1000
    - link/ether f6:b7:10:ec:cf:2e brd ff:ff:ff:ff:ff:ff
    - inet 10.0.0.1/24 scope global veth-a
    - valid_lft forever preferred_lft forever
    - inet6 fe80::f4b7:10ff:feec:cf2e/64 scope link proto kernel_ll
    - valid_lft forever preferred_lft forever
    - 인터페이스 이름 veth-a과 함께 이 인터페이스가 veth-b와 가상 랜선으로 서로 연결된 상태이다.
    - STATE UP은 해당 인터페이스가 활성화되어있는 상태임을 의미한다.
        - ↔ STAND DOWN
- link/ether ... : veth-a의 하드웨어 주소인 MAC 주소이다.
- inet 10.0.0.1/24 는 할당되어있는 IPv4 주소와 서브넷 마스크이다. (이는 네트워크 계층을 사용할 때 = 서브넷마스크가 다를 때 이용한다)